# Model Weights and Tokens: Numbers All the Way Down
## Using llama-cpp-python and the GGUF Format

This notebook pulls back the curtain on **how a language model actually works with numbers**. We'll explore:

- **Tokens**: How text is broken into numeric pieces before the model ever sees it
- **Tokens going in and out**: The integer IDs that flow through a model
- **Text chunks**: What a chunk of text looks like vs. its tokenized form
- **GGUF files**: The binary container that stores a model on disk
- **Model weights**: The actual floating-point numbers that give a model its "knowledge"
- **How concepts map to numbers**: Token embedding vectors as coordinates in meaning-space

Throughout, we use **economics examples** — tariffs, inflation, GDP, supply and demand — as the text to analyze, since these are rich, domain-specific phrases that illustrate how tokenizers handle specialized vocabulary.

### What You'll Need

- A `.gguf` model file already downloaded (see `GPT4All_Download_gguf.ipynb`)
- Python packages: `llama-cpp-python`, `gguf`, `numpy`

### Resources

- [GGUF specification](https://github.com/ggerganov/ggml/blob/master/docs/gguf.md)
- [llama-cpp-python docs](https://llama-cpp-python.readthedocs.io/)
- [The Illustrated Word2Vec](https://jalammar.github.io/illustrated-word2vec/) — great visual intro to embeddings
- [Tokenizer visualization](https://tiktokenizer.vercel.app/) — explore GPT tokenization online

### Attribution

Notebook developed by Eric Van Dusen and contributors for the ds-modules/SmallLM-FA25 project.

---
## 1. Environment Setup

We need two packages beyond the standard library:

| Package | Purpose |
|---------|--------|
| `llama-cpp-python` | Load GGUF models, run inference, access the built-in tokenizer |
| `gguf` | Read GGUF file metadata and weight tensors directly (no model load needed) |
| `numpy` | Display weight arrays as readable tables |

The `gguf` package is the official Python reader published by the llama.cpp project.

In [1]:
# Install llama-cpp-python if not already present
try:
    from llama_cpp import Llama
except ImportError:
    %pip install llama-cpp-python
    from llama_cpp import Llama

# Install the gguf reader package
try:
    from gguf import GGUFReader
except ImportError:
    %pip install gguf
    from gguf import GGUFReader

import numpy as np
print("All packages loaded successfully!")

Note: you may need to restart the kernel to use updated packages.
All packages loaded successfully!


### 1.1 Locate the Model File

Set `model_path` and `model_name` to match your environment. Common locations:

- **Shared JupyterHub**: `/home/jovyan/shared/`
- **Local machine**: your own path (e.g. `~/models/`)

In [3]:
# ── Set these two variables to match your setup ─────────────────────────────
#model_path = "/home/jovyan/shared/"           # directory containing .gguf files
model_name = "qwen2-1_5b-instruct-q4_0.gguf" # filename of the model
# ─────────────────────────────────────────────────────────────────────────────

# For local use, uncomment and adjust:
# model_path = "shared-rw/"
model_path = "/Users/ericvandusen/SmallLM/Models/"

import os
full_model_path = os.path.join(model_path, model_name)
print(f"Looking for: {full_model_path}")
print(f"File exists: {os.path.exists(full_model_path)}")

Looking for: /Users/ericvandusen/SmallLM/Models/qwen2-1_5b-instruct-q4_0.gguf
File exists: True


In [4]:
# See what .gguf files are available
gguf_files = [f for f in os.listdir(model_path) if f.endswith(".gguf")]
print(f"Available .gguf models in {model_path}:")
for f in gguf_files:
    size_mb = os.path.getsize(os.path.join(model_path, f)) / (1024**2)
    print(f"  {f:60s}  {size_mb:7.1f} MB")

Available .gguf models in /Users/ericvandusen/SmallLM/Models/:
  mistral-7b-instruct-v0.1.Q4_0.gguf                              397.0 MB
  qwen2-1_5b-instruct-q4_0.gguf                                   894.1 MB
  gemma-2b-it.Q4_0.gguf                                             0.0 MB
  DeepSeek-R1-Distill-Qwen-1.5B-Q4_0.gguf                        1019.3 MB
  orca-mini-3b-gguf2-q4_0.gguf                                   1888.2 MB


---
## 2. Motivation: Everything Is a Number

Before we load the model, let's build intuition. Consider this economics sentence:

> *"When a country imposes a tariff on imported steel, domestic producers gain while consumers pay higher prices."*

A language model **never sees letters**. It sees only:

1. **Token IDs** — integers that index a vocabulary table  
2. **Embedding vectors** — lists of floating-point numbers that represent each token  
3. **Weight matrices** — huge arrays of floats that transform those vectors step by step  

The diagram below summarizes the pipeline:

```
Raw text  →  Tokenizer  →  Token IDs  →  Embedding lookup  →  Transformer layers  →  Logits  →  Next token
 (str)                    (list[int])     (matrix of floats)     (weight math)        (floats)    (int → str)
```

Every step in this pipeline is **pure arithmetic on numbers**. This section walks through each step using economic text.

---
## 3. Tokens: Text Broken Into Pieces

### What is a Token?

A **token** is the atomic unit of text that the model processes. Tokens are not the same as words:

- A common short word (e.g. `the`) is usually one token.
- A longer or rarer word may be split into sub-word pieces (e.g. `inflation` → `[inflation]`, but `hyperinflation` → `[hyper, inflation]`).
- Spaces, punctuation, and capitalization all affect tokenization.
- Numbers can be tokenized digit-by-digit or as whole numbers depending on the tokenizer.

Modern LLMs typically use **Byte-Pair Encoding (BPE)** or **SentencePiece** tokenizers. Both learn a vocabulary of common sub-word units from a large training corpus.

### Why Does This Matter for Economics?

Economics has technical vocabulary that may be rare in general text: *amortization*, *monopsony*, *heteroscedasticity*. A tokenizer trained on general web data may split these into many small pieces, giving the model less efficient representations of economic concepts.

### 3.1 Load the Model and Access Its Tokenizer

We load the model with `verbose=False` to suppress the C++ startup messages. Once loaded, the `Llama` object exposes a `tokenize()` method that runs the **same tokenizer** used during training.

In [5]:
# Load the model
# verbose=False silences the C++ loading messages
model = Llama(
    model_path=full_model_path,
    n_ctx=2048,
    verbose=False
)
print(f"Model loaded: {model_name}")
print(f"Vocabulary size: {model.n_vocab()} tokens")
print(f"Context window:  {model.n_ctx()} tokens")

llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
ggml_metal_init: skipping kernel_get_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_set_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_c4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_1row              (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_l4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_bf16                  (not supported)
ggml_metal_init: skipping kernel_mul_mv_id_bf16_f32                (not supported)
ggml_metal_init: skipping kernel_mul_mm_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mm_id_bf16_f16                (not supported)
ggml_metal_init: skipping kernel_flash_attn_ext_bf16_h64 

Model loaded: qwen2-1_5b-instruct-q4_0.gguf
Vocabulary size: 151936 tokens
Context window:  2048 tokens


### 3.2 Tokenizing Economics Text

Let's tokenize a few economics sentences and see what comes out. The `tokenize()` method returns a Python list of integers — the **token IDs**.

In [6]:
# Economics sentences to tokenize
economics_sentences = [
    "Comparative advantage explains why countries specialize in producing goods they can make at a lower opportunity cost.",
    "When a central bank raises interest rates, borrowing becomes more expensive and inflation tends to fall.",
    "GDP measures the total monetary value of all finished goods and services produced within a country.",
    "A tariff is a tax on imported goods that raises their price for domestic consumers.",
    "Supply and demand determine the equilibrium price in a competitive market.",
]

# Tokenize each sentence and display the result
for sentence in economics_sentences:
    # encode the text to bytes first (llama_cpp expects bytes)
    token_ids = model.tokenize(sentence.encode("utf-8"))
    print(f"Text  : {sentence[:70]}..." if len(sentence) > 70 else f"Text  : {sentence}")
    print(f"Tokens: {token_ids}")
    print(f"Count : {len(token_ids)} tokens for {len(sentence)} characters  "
          f"(ratio: {len(sentence)/len(token_ids):.1f} chars/token)")
    print()

Text  : Comparative advantage explains why countries specialize in producing g...
Tokens: [17123, 1388, 9423, 14758, 3170, 5837, 47344, 304, 17387, 11561, 807, 646, 1281, 518, 264, 4722, 6638, 2783, 13]
Count : 19 tokens for 117 characters  (ratio: 6.2 chars/token)

Text  : When a central bank raises interest rates, borrowing becomes more expe...
Tokens: [4498, 264, 8622, 6073, 24886, 2734, 7813, 11, 52575, 9044, 803, 11392, 323, 24527, 27236, 311, 4399, 13]
Count : 18 tokens for 104 characters  (ratio: 5.8 chars/token)

Text  : GDP measures the total monetary value of all finished goods and servic...
Tokens: [38, 10298, 10953, 279, 2790, 32284, 897, 315, 678, 8060, 11561, 323, 3516, 8947, 2878, 264, 3146, 13]
Count : 18 tokens for 99 characters  (ratio: 5.5 chars/token)

Text  : A tariff is a tax on imported goods that raises their price for domest...
Tokens: [32, 60261, 374, 264, 3742, 389, 24928, 11561, 429, 24886, 862, 3349, 369, 12728, 13417, 13]
Count : 16 tokens for 83 character

### 3.3 Tokens Going In: The Integer Stream

When you call a model with a prompt, the model receives **a list of integers** — not text. Let's look closely at one sentence and see exactly which token IDs correspond to which pieces of text.

In [7]:
# Pick one sentence to examine closely
example_text = "A tariff is a tax on imported goods that raises their price for domestic consumers."

token_ids = model.tokenize(example_text.encode("utf-8"))

print("=" * 60)
print("TOKENS GOING IN")
print("=" * 60)
print(f"Input text: '{example_text}'")
print(f"\nToken IDs (the integers the model actually sees):")
print(token_ids)
print(f"\nTotal: {len(token_ids)} tokens")

TOKENS GOING IN
Input text: 'A tariff is a tax on imported goods that raises their price for domestic consumers.'

Token IDs (the integers the model actually sees):
[32, 60261, 374, 264, 3742, 389, 24928, 11561, 429, 24886, 862, 3349, 369, 12728, 13417, 13]

Total: 16 tokens


In [8]:
# Now decode each token ID back to text to see what each integer represents
print(f"{'Token ID':>10}  {'Text piece':30}")
print("-" * 45)
for token_id in token_ids:
    # detokenize a single token
    piece_bytes = model.detokenize([token_id])
    piece_str = piece_bytes.decode("utf-8", errors="replace")
    print(f"{token_id:>10}  {repr(piece_str):30}")

  Token ID  Text piece                    
---------------------------------------------
        32  'A'                           
     60261  ' tariff'                     
       374  ' is'                         
       264  ' a'                          
      3742  ' tax'                        
       389  ' on'                         
     24928  ' imported'                   
     11561  ' goods'                      
       429  ' that'                       
     24886  ' raises'                     
       862  ' their'                      
      3349  ' price'                      
       369  ' for'                        
     12728  ' domestic'                   
     13417  ' consumers'                  
        13  '.'                           


**Notice a few things:**
- Spaces are usually attached to the *beginning* of the next word token (you'll see `' tariff'` not `'tariff'`).
- Punctuation gets its own token.
- Common short words (`a`, `is`, `on`) are single tokens with small IDs (frequent words get low IDs in BPE).
- Words like `imported` may be one token because they're common enough, while rare words would split.

### 3.4 Tokens Going Out: Generating New Token IDs

When the model *generates* text, it also produces **token IDs first**, then decodes them to text. Let's capture the output token IDs directly.

In [9]:
# Run inference and capture token-level output
prompt = "Define comparative advantage in one sentence:"

print("=" * 60)
print("TOKENS GOING OUT")
print("=" * 60)
print(f"Prompt: '{prompt}'")
print()

# Use the low-level generate() method to get token IDs
prompt_tokens = model.tokenize(prompt.encode("utf-8"))
print(f"Prompt token IDs ({len(prompt_tokens)} tokens): {prompt_tokens}")
print()

# Generate up to 60 output tokens
output_tokens = []
output_text_pieces = []

for token_id in model.generate(prompt_tokens, top_k=1):  # top_k=1: greedy decoding — always pick the single highest-probability token
    output_tokens.append(token_id)
    piece = model.detokenize([token_id]).decode("utf-8", errors="replace")
    output_text_pieces.append(piece)
    
    # Stop at EOS or after 60 tokens
    if token_id == model.token_eos() or len(output_tokens) >= 60:
        break

print(f"Output token IDs ({len(output_tokens)} tokens):")
print(output_tokens)
print()
print("Decoded output:")
print("".join(output_text_pieces))

TOKENS GOING OUT
Prompt: 'Define comparative advantage in one sentence:'

Prompt token IDs (7 tokens): [35338, 54294, 9423, 304, 825, 11652, 25]

Output token IDs (60 tokens):
[96180, 9423, 374, 279, 5726, 311, 8193, 264, 1661, 476, 2473, 803, 29720, 1091, 3800, 11, 892, 6147, 264, 3146, 311, 8193, 803, 315, 429, 1661, 476, 2473, 518, 264, 4722, 2783, 13, 1096, 9423, 646, 387, 1483, 311, 6559, 448, 1008, 5837, 11, 1380, 279, 3146, 448, 279, 54294, 9423, 646, 8193, 803, 315, 279, 1661, 476, 2473, 518]

Decoded output:
 Comparative advantage is the ability to produce a good or service more efficiently than others, which allows a country to produce more of that good or service at a lower cost. This advantage can be used to trade with other countries, where the country with the comparative advantage can produce more of the good or service at


In [10]:
# Show output token-by-token in a table
print(f"{'Step':>6}  {'Token ID':>10}  {'Text piece'}")
print("-" * 40)
for i, (tid, piece) in enumerate(zip(output_tokens, output_text_pieces)):
    print(f"{i+1:>6}  {tid:>10}  {repr(piece)}")

  Step    Token ID  Text piece
----------------------------------------
     1       96180  ' Comparative'
     2        9423  ' advantage'
     3         374  ' is'
     4         279  ' the'
     5        5726  ' ability'
     6         311  ' to'
     7        8193  ' produce'
     8         264  ' a'
     9        1661  ' good'
    10         476  ' or'
    11        2473  ' service'
    12         803  ' more'
    13       29720  ' efficiently'
    14        1091  ' than'
    15        3800  ' others'
    16          11  ','
    17         892  ' which'
    18        6147  ' allows'
    19         264  ' a'
    20        3146  ' country'
    21         311  ' to'
    22        8193  ' produce'
    23         803  ' more'
    24         315  ' of'
    25         429  ' that'
    26        1661  ' good'
    27         476  ' or'
    28        2473  ' service'
    29         518  ' at'
    30         264  ' a'
    31        4722  ' lower'
    32        2783  ' cost'
    33          1

**Key insight**: The model generates **one integer at a time**. Each output integer is a token ID from its vocabulary. Only at the end does the application decode those IDs back to readable text. The LLM is, at its core, a machine that predicts the next integer in a sequence.

---
## 4. What Does a Chunk Look Like?

### Text Chunks

In real applications (like Retrieval-Augmented Generation, or RAG), long documents are **chunked** — split into overlapping windows — before being fed to a model. A **chunk** is just a substring of the source text, small enough to fit in the model's context window.

Below we chunk an economics paragraph by character count (a simple strategy) and then tokenize each chunk.

In [11]:
# A longer economics passage to chunk
economics_passage = """
Comparative advantage is one of the most important concepts in international trade theory, 
first articulated by David Ricardo in 1817. The principle states that even if one country is 
more efficient at producing all goods than another, both countries can still benefit from 
specialization and trade. The key insight is opportunity cost: a country should specialize 
in producing goods where its relative efficiency advantage is greatest, or its relative 
inefficiency is smallest. For example, if the United States can produce both wheat and 
semiconductors more efficiently than Vietnam, but its advantage in semiconductors is 
proportionally larger, then the US should specialize in semiconductors and import wheat 
from Vietnam. Both countries end up with more of both goods than if each tried to produce 
everything domestically. This principle underpins the argument for free trade and explains 
much of the pattern of international specialization we observe in the global economy.
""".strip()

print(f"Passage length: {len(economics_passage)} characters")
print()
print(economics_passage)

Passage length: 989 characters

Comparative advantage is one of the most important concepts in international trade theory, 
first articulated by David Ricardo in 1817. The principle states that even if one country is 
more efficient at producing all goods than another, both countries can still benefit from 
specialization and trade. The key insight is opportunity cost: a country should specialize 
in producing goods where its relative efficiency advantage is greatest, or its relative 
inefficiency is smallest. For example, if the United States can produce both wheat and 
semiconductors more efficiently than Vietnam, but its advantage in semiconductors is 
proportionally larger, then the US should specialize in semiconductors and import wheat 
from Vietnam. Both countries end up with more of both goods than if each tried to produce 
everything domestically. This principle underpins the argument for free trade and explains 
much of the pattern of international specialization we observe i

In [12]:
def chunk_text(text, chunk_size=200, overlap=50):
    """
    Split text into overlapping chunks of roughly chunk_size characters.
    overlap: how many characters from the end of one chunk appear at the start of the next.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(economics_passage, chunk_size=200, overlap=40)

for i, chunk in enumerate(chunks):
    print(f"── Chunk {i+1} ({len(chunk)} chars) ──────────────────────────")
    print(chunk)
    print()

── Chunk 1 (200 chars) ──────────────────────────
Comparative advantage is one of the most important concepts in international trade theory, 
first articulated by David Ricardo in 1817. The principle states that even if one country is 
more efficient

── Chunk 2 (200 chars) ──────────────────────────
t even if one country is 
more efficient at producing all goods than another, both countries can still benefit from 
specialization and trade. The key insight is opportunity cost: a country should spe

── Chunk 3 (200 chars) ──────────────────────────
s opportunity cost: a country should specialize 
in producing goods where its relative efficiency advantage is greatest, or its relative 
inefficiency is smallest. For example, if the United States ca

── Chunk 4 (200 chars) ──────────────────────────
st. For example, if the United States can produce both wheat and 
semiconductors more efficiently than Vietnam, but its advantage in semiconductors is 
proportionally larger, then the US should 

### 4.1 What Does a Tokenized Chunk Look Like?

Now let's take the same chunks and tokenize them. This shows exactly what the model receives as input — not sentences or words, but **lists of integers**.

In [13]:
for i, chunk in enumerate(chunks):
    token_ids = model.tokenize(chunk.encode("utf-8"))
    print(f"── Chunk {i+1} ────────────────────────────────────────")
    print(f"  Text  ({len(chunk):3d} chars):  {chunk[:80]}..." if len(chunk) > 80 else 
          f"  Text  ({len(chunk):3d} chars):  {chunk}")
    print(f"  Token IDs ({len(token_ids):2d} tokens): {token_ids}")
    print(f"  Chars-per-token ratio: {len(chunk)/len(token_ids):.2f}")
    print()

── Chunk 1 ────────────────────────────────────────
  Text  (200 chars):  Comparative advantage is one of the most important concepts in international tra...
  Token IDs (40 tokens): [17123, 1388, 9423, 374, 825, 315, 279, 1429, 2989, 18940, 304, 6489, 6559, 10126, 11, 715, 3896, 82180, 553, 6798, 65950, 304, 220, 16, 23, 16, 22, 13, 576, 17508, 5302, 429, 1496, 421, 825, 3146, 374, 715, 6384, 11050]
  Chars-per-token ratio: 5.00

── Chunk 2 ────────────────────────────────────────
  Text  (200 chars):  t even if one country is 
more efficient at producing all goods than another, bo...
  Token IDs (39 tokens): [83, 1496, 421, 825, 3146, 374, 715, 6384, 11050, 518, 17387, 678, 11561, 1091, 2441, 11, 2176, 5837, 646, 2058, 8760, 504, 715, 15144, 2022, 323, 6559, 13, 576, 1376, 20017, 374, 6638, 2783, 25, 264, 3146, 1265, 2341]
  Chars-per-token ratio: 5.13

── Chunk 3 ────────────────────────────────────────
  Text  (200 chars):  s opportunity cost: a country should specialize 
in produc

In [14]:
# Side-by-side comparison: raw chunk vs tokenized chunk (first chunk only)
chunk = chunks[0]
token_ids = model.tokenize(chunk.encode("utf-8"))

print("RAW TEXT CHUNK:")
print("-" * 60)
print(chunk)
print()

print("TOKENIZED CHUNK (token IDs):")
print("-" * 60)
print(token_ids)
print()

print("TOKEN → TEXT MAPPING (first 20 tokens):")
print("-" * 60)
print(f"{'ID':>8}  {'Text piece'}")
for tid in token_ids[:20]:
    piece = model.detokenize([tid]).decode("utf-8", errors="replace")
    print(f"{tid:>8}  {repr(piece)}")
if len(token_ids) > 20:
    print(f"    ... ({len(token_ids)-20} more tokens)")

RAW TEXT CHUNK:
------------------------------------------------------------
Comparative advantage is one of the most important concepts in international trade theory, 
first articulated by David Ricardo in 1817. The principle states that even if one country is 
more efficient

TOKENIZED CHUNK (token IDs):
------------------------------------------------------------
[17123, 1388, 9423, 374, 825, 315, 279, 1429, 2989, 18940, 304, 6489, 6559, 10126, 11, 715, 3896, 82180, 553, 6798, 65950, 304, 220, 16, 23, 16, 22, 13, 576, 17508, 5302, 429, 1496, 421, 825, 3146, 374, 715, 6384, 11050]

TOKEN → TEXT MAPPING (first 20 tokens):
------------------------------------------------------------
      ID  Text piece
   17123  'Compar'
    1388  'ative'
    9423  ' advantage'
     374  ' is'
     825  ' one'
     315  ' of'
     279  ' the'
    1429  ' most'
    2989  ' important'
   18940  ' concepts'
     304  ' in'
    6489  ' international'
    6559  ' trade'
   10126  ' theory'
      11  ','
  

---
## 5. Inside a GGUF File

### What is GGUF?

**GGUF** (GGML Universal File Format) is the binary file format used to store quantized models. A single `.gguf` file contains everything needed to run a model:

1. **File header** — magic bytes, version number
2. **Key-value metadata** — model architecture, hyperparameters, tokenizer vocabulary
3. **Tensor data** — the actual weight matrices, stored as quantized floats

Think of it as a self-describing archive: you can learn almost everything about a model just by reading its GGUF file, without loading it into a neural-network framework.

The Python `gguf` package (from the llama.cpp project) lets us read this file directly.

### 5.1 Reading GGUF Metadata

The metadata section stores configuration values — things like the number of attention heads, the hidden dimension size, the context length, and the vocabulary.

In [15]:
from gguf import GGUFReader

# Open the GGUF file without loading it into GPU/CPU as a model
reader = GGUFReader(full_model_path)

print(f"GGUF file: {full_model_path}")
print(f"File size: {os.path.getsize(full_model_path) / (1024**2):.1f} MB")
print()
print(f"Number of metadata key-value pairs: {len(reader.fields)}")
print(f"Number of weight tensors:           {len(reader.tensors)}")

GGUF file: /Users/ericvandusen/SmallLM/Models/qwen2-1_5b-instruct-q4_0.gguf
File size: 894.1 MB

Number of metadata key-value pairs: 29
Number of weight tensors:           338


In [16]:
# Print all metadata fields
print("=" * 70)
print("GGUF METADATA (key-value pairs)")
print("=" * 70)

for name, field in reader.fields.items():
    # field.parts contains the raw data; field.data gives usable values
    try:
        if len(field.data) == 1:
            value = field.data[0]
            # Decode bytes to string if needed
            if isinstance(value, (bytes, bytearray)):
                value = value.decode("utf-8", errors="replace")
        else:
            value = f"[array of {len(field.data)} items]"
    except Exception:
        value = "<unreadable>"
    
    # Skip huge tokenizer arrays for now (we'll look at them separately)
    if "token" in name.lower() and isinstance(value, str) and value.startswith("["):
        value = "[vocabulary array — shown below]"
    
    print(f"  {name:<50} = {str(value)[:60]}")

GGUF METADATA (key-value pairs)
  GGUF.version                                       = 0
  GGUF.tensor_count                                  = 0
  GGUF.kv_count                                      = 0
  general.architecture                               = 4
  general.name                                       = 4
  qwen2.block_count                                  = 3
  qwen2.context_length                               = 3
  qwen2.embedding_length                             = 3
  qwen2.feed_forward_length                          = 3
  qwen2.attention.head_count                         = 3
  qwen2.attention.head_count_kv                      = 3
  qwen2.rope.freq_base                               = 3
  qwen2.attention.layer_norm_rms_epsilon             = 3
  general.file_type                                  = 3
  tokenizer.ggml.model                               = 4
  tokenizer.ggml.pre                                 = 4
  tokenizer.ggml.tokens                              = [

In [17]:
# Highlight the architecture-defining fields
architecture_keys = [
    "general.architecture",
    "general.name",
    "general.parameter_count",
    "general.quantization_version",
]

# Collect keys that mention important dimensions
dimension_keywords = ["hidden", "head", "layer", "context", "embed", "ff", "feed",
                      "attention", "block", "n_head", "n_layer", "n_ctx", "n_embd"]

print("=" * 70)
print("KEY ARCHITECTURAL PARAMETERS")
print("=" * 70)

for name, field in reader.fields.items():
    if any(kw in name.lower() for kw in dimension_keywords) or name in architecture_keys:
        try:
            if len(field.data) == 1:
                value = field.data[0]
                if isinstance(value, (bytes, bytearray)):
                    value = value.decode("utf-8", errors="replace")
            else:
                value = f"[{len(field.data)} items]"
        except Exception:
            value = "<unreadable>"
        print(f"  {name:<50} = {value}")

KEY ARCHITECTURAL PARAMETERS
  general.architecture                               = 4
  general.name                                       = 4
  qwen2.block_count                                  = 3
  qwen2.context_length                               = 3
  qwen2.embedding_length                             = 3
  qwen2.feed_forward_length                          = 3
  qwen2.attention.head_count                         = 3
  qwen2.attention.head_count_kv                      = 3
  qwen2.attention.layer_norm_rms_epsilon             = 3
  general.quantization_version                       = 3


**What those numbers mean:**

| Metadata key | Meaning |
|---|---|
| `*.embedding_length` | Size of each token's embedding vector (e.g. 1536 = 1536 floats per token) |
| `*.block_count` | Number of transformer layers stacked on top of each other |
| `*.attention.head_count` | Number of parallel attention heads per layer |
| `*.context_length` | Maximum number of tokens the model can see at once |
| `*.feed_forward_length` | Size of the intermediate layer inside each transformer block |

These numbers determine the **capacity** (and the size) of the model.

### 5.2 The Tokenizer Vocabulary Inside the GGUF File

The vocabulary — all the text pieces the tokenizer knows — is stored directly in the GGUF file. Let's look at the first and last few entries to see what the token table looks like.

In [18]:
# Find the tokenizer vocabulary field
vocab_field = None
for name, field in reader.fields.items():
    if "tokens" in name.lower() and "tokenizer" in name.lower():
        vocab_field = (name, field)
        break

if vocab_field is None:
    print("Vocabulary field not found — trying alternative names...")
    for name, field in reader.fields.items():
        if "vocab" in name.lower() or "token" in name.lower():
            print(f"  Found: {name} ({len(field.data)} entries)")
else:
    name, field = vocab_field
    print(f"Vocabulary field: '{name}'")
    print(f"Total vocabulary size: {len(field.data)} tokens")
    print()
    
    # Show first 20 tokens
    print("First 20 tokens (low IDs = most common in training data):")
    print(f"{'Token ID':>10}  {'Token text'}")
    print("-" * 35)
    for i in range(min(20, len(field.data))):
        token_bytes = field.data[i]
        if isinstance(token_bytes, (bytes, bytearray, memoryview)):
            token_text = bytes(token_bytes).decode("utf-8", errors="replace")
        else:
            token_text = str(token_bytes)
        print(f"{i:>10}  {repr(token_text)}")
    
    print()
    print("Last 10 tokens (high IDs = rare or special tokens):")
    print(f"{'Token ID':>10}  {'Token text'}")
    print("-" * 35)
    total = len(field.data)
    for i in range(total - 10, total):
        token_bytes = field.data[i]
        if isinstance(token_bytes, (bytes, bytearray, memoryview)):
            token_text = bytes(token_bytes).decode("utf-8", errors="replace")
        else:
            token_text = str(token_bytes)
        print(f"{i:>10}  {repr(token_text)}")

Vocabulary field: 'tokenizer.ggml.tokens'
Total vocabulary size: 151936 tokens

First 20 tokens (low IDs = most common in training data):
  Token ID  Token text
-----------------------------------
         0  '6'
         1  '8'
         2  '10'
         3  '12'
         4  '14'
         5  '16'
         6  '18'
         7  '20'
         8  '22'
         9  '24'
        10  '26'
        11  '28'
        12  '30'
        13  '32'
        14  '34'
        15  '36'
        16  '38'
        17  '40'
        18  '42'
        19  '44'

Last 10 tokens (high IDs = rare or special tokens):
  Token ID  Token text
-----------------------------------
    151926  '303858'
    151927  '303860'
    151928  '303862'
    151929  '303864'
    151930  '303866'
    151931  '303868'
    151932  '303870'
    151933  '303872'
    151934  '303874'
    151935  '303876'


### 5.3 Listing All Weight Tensors

Now let's look at all the **weight tensors** stored in the GGUF file. Each tensor has:
- A **name** (tells you which layer and what type of weight)
- A **shape** (dimensions of the matrix)
- A **type** (quantization format: Q4_0, Q8_0, F16, etc.)
- A **size** (bytes on disk — much smaller than the raw float32 equivalent)

In [19]:
print("=" * 80)
print("WEIGHT TENSORS IN THE GGUF FILE")
print("=" * 80)
print(f"{'Tensor name':<50} {'Shape':<25} {'Type':<10}")
print("-" * 85)

for tensor in reader.tensors:
    shape_str = str(list(tensor.shape))
    print(f"{tensor.name:<50} {shape_str:<25} {str(tensor.tensor_type.name):<10}")

print()
print(f"Total: {len(reader.tensors)} tensors")

WEIGHT TENSORS IN THE GGUF FILE
Tensor name                                        Shape                     Type      
-------------------------------------------------------------------------------------
token_embd.weight                                  [np.uint64(1536), np.uint64(151936)] Q6_K      
blk.0.attn_norm.weight                             [np.uint64(1536)]         F32       
blk.0.ffn_down.weight                              [np.uint64(8960), np.uint64(1536)] Q4_1      
blk.0.ffn_gate.weight                              [np.uint64(1536), np.uint64(8960)] Q4_0      
blk.0.ffn_up.weight                                [np.uint64(1536), np.uint64(8960)] Q4_0      
blk.0.ffn_norm.weight                              [np.uint64(1536)]         F32       
blk.0.attn_k.bias                                  [np.uint64(256)]          F32       
blk.0.attn_k.weight                                [np.uint64(1536), np.uint64(256)] Q4_0      
blk.0.attn_output.weight                    

**Reading the tensor names** — the naming convention follows a pattern:

| Name pattern | Meaning |
|---|---|
| `token_embd.weight` | The **embedding matrix** — one row per vocabulary token |
| `blk.N.attn_q.weight` | **Query** weight matrix for attention in layer N |
| `blk.N.attn_k.weight` | **Key** weight matrix for attention in layer N |
| `blk.N.attn_v.weight` | **Value** weight matrix for attention in layer N |
| `blk.N.attn_output.weight` | Attention **output projection** in layer N |
| `blk.N.ffn_up.weight` | Feed-forward network **up-projection** in layer N |
| `blk.N.ffn_down.weight` | Feed-forward network **down-projection** in layer N |
| `blk.N.ffn_gate.weight` | Feed-forward **gate** (SwiGLU activation) in layer N |
| `blk.N.attn_norm.weight` | Layer **normalization** scale in attention block N |
| `output.weight` | Final **output (lm head)** matrix that maps to vocabulary logits |

The model processes tokens by running them through all `N` blocks in sequence.

In [21]:
# Count tensor types to understand the model structure
from collections import Counter

# Categorize tensors by their role
categories = {
    "embedding": [],
    "attention_q": [],
    "attention_k": [],
    "attention_v": [],
    "attention_output": [],
    "feed_forward": [],
    "normalization": [],
    "output": [],
    "other": []
}

for tensor in reader.tensors:
    n = tensor.name
    if "token_embd" in n:
        categories["embedding"].append(n)
    elif "attn_q" in n:
        categories["attention_q"].append(n)
    elif "attn_k" in n:
        categories["attention_k"].append(n)
    elif "attn_v" in n:
        categories["attention_v"].append(n)
    elif "attn_output" in n or "attn_out" in n:
        categories["attention_output"].append(n)
    elif "ffn" in n:
        categories["feed_forward"].append(n)
    elif "norm" in n:
        categories["normalization"].append(n)
    elif n in ("output.weight", "output_norm.weight"):
        categories["output"].append(n)
    else:
        categories["other"].append(n)

print("Tensor breakdown by role:")
print(f"{'Role':<25} {'Count':>6}")
print("-" * 33)
for role, tensors in categories.items():
    if tensors:
        print(f"{role:<25} {len(tensors):>6}")
print("-" * 33)
print(f"{'TOTAL':<25} {len(reader.tensors):>6}")

Tensor breakdown by role:
Role                       Count
---------------------------------
embedding                      1
attention_q                   56
attention_k                   56
attention_v                   56
attention_output              28
feed_forward                 112
normalization                 29
---------------------------------
TOTAL                        338


---
## 6. Model Weights as Numbers

Now let's actually look at the **numbers** inside the weight tensors.

### 6.1 The Token Embedding Matrix

The embedding matrix is the most conceptually important tensor:
- It has one **row per token** in the vocabulary
- Each row is a **vector of floats** (the "embedding") for that token
- This is how abstract token IDs get turned into rich numeric representations

Shape: `[vocab_size  ×  embedding_dim]`  
Example: `[151936 × 1536]` for Qwen2-1.5B

In [22]:
# Find the token embedding tensor
embd_tensor = None
for tensor in reader.tensors:
    if tensor.name == "token_embd.weight":
        embd_tensor = tensor
        break

if embd_tensor is not None:
    print(f"Tensor name:  {embd_tensor.name}")
    print(f"Shape:        {list(embd_tensor.shape)}")
    print(f"  → {embd_tensor.shape[0]} tokens in vocabulary")
    print(f"  → {embd_tensor.shape[1]} floats per token embedding")
    print(f"Storage type: {embd_tensor.tensor_type.name}  (quantized to save space)")
    
    vocab_size   = embd_tensor.shape[0]
    embd_dim     = embd_tensor.shape[1]
    full_fp32_mb = vocab_size * embd_dim * 4 / (1024**2)
    print(f"\nIf stored as full float32: {full_fp32_mb:.1f} MB")
    print(f"(Quantization compresses this significantly)")
else:
    print("Embedding tensor not found — check tensor names above.")

Tensor name:  token_embd.weight
Shape:        [np.uint64(1536), np.uint64(151936)]
  → 1536 tokens in vocabulary
  → 151936 floats per token embedding
Storage type: Q6_K  (quantized to save space)

If stored as full float32: 890.2 MB
(Quantization compresses this significantly)


In [23]:
# Read the raw quantized data from the embedding tensor
# GGUFReader stores the raw bytes; we dequantize to float32 for display
if embd_tensor is not None:
    # Convert to float32 numpy array
    raw_data = embd_tensor.data          # numpy array (possibly quantized uint8)
    
    print(f"Raw data shape (quantized storage): {raw_data.shape}")
    print(f"Raw data dtype: {raw_data.dtype}")
    print()
    print("First 20 raw quantized bytes:")
    print(raw_data.flat[:20])
    print()
    print("(These bytes encode groups of 32 float values in a compact quantized format.)")

Raw data shape (quantized storage): (151936, 1260)
Raw data dtype: uint8

First 20 raw quantized bytes:
[ 24 251  76  96  12 240  63  19 128 240  88  69 133  24 209 133  15 144
  83 172]

(These bytes encode groups of 32 float values in a compact quantized format.)


### 6.2 Dequantizing to See Actual Float Values

GGUF stores weights in a **quantized** format to save space. Q4_0 quantization packs 32 float values into 18 bytes (instead of 128 bytes for float32) by storing them as 4-bit integers with a shared scaling factor.

To see the actual float values, we need to dequantize. The `llama_cpp` library does this automatically when running inference, but we can also do it manually using the `gguf` package utilities.

In [24]:
# Attempt to dequantize using numpy (manual Q4_0 dequantization)
# Q4_0 format: each block of 32 weights = 1 fp16 scale + 16 bytes of 4-bit ints
# Block size in bytes: 2 (scale) + 16 (4-bit data) = 18 bytes per 32 weights

if embd_tensor is not None and str(embd_tensor.tensor_type.name) == "Q4_0":
    raw = embd_tensor.data   # shape: (n_bytes,) uint8
    
    # Each Q4_0 block = 18 bytes = 2 bytes (fp16 scale) + 16 bytes (4-bit * 32 weights)
    block_size_bytes = 18
    weights_per_block = 32
    n_blocks = len(raw) // block_size_bytes
    
    print(f"Dequantizing Q4_0 embedding matrix...")
    print(f"  Total blocks: {n_blocks}")
    print(f"  Total weights: {n_blocks * weights_per_block:,}")
    print()
    
    # Dequantize first few blocks so we can display the actual float values
    blocks_to_show = min(4, n_blocks)  # just first 4 blocks = 128 weights
    decoded_weights = []
    
    for b in range(blocks_to_show):
        block = raw[b * block_size_bytes : (b + 1) * block_size_bytes]
        # Extract fp16 scale (first 2 bytes)
        scale = np.frombuffer(block[:2], dtype=np.float16)[0].astype(np.float32)
        # Extract 4-bit quantized values (remaining 16 bytes = 32 values)
        quant_bytes = block[2:]
        lo = (quant_bytes & 0x0F).astype(np.int8)
        hi = (quant_bytes >> 4).astype(np.int8)
        # Values are stored as signed 4-bit: range -8..7
        nibbles = np.empty(32, dtype=np.int8)
        nibbles[0::2] = lo - 8
        nibbles[1::2] = hi - 8
        dequant = nibbles.astype(np.float32) * scale
        decoded_weights.append((scale, dequant))
    
    # Display block 0 (first 32 weights of the first token's embedding)
    scale0, weights0 = decoded_weights[0]
    print(f"First Q4_0 block of the embedding matrix:")
    print(f"  Scale factor: {scale0:.6f}")
    print(f"  32 dequantized float values:")
    print(np.round(weights0, 5))

elif embd_tensor is not None:
    ttype = str(embd_tensor.tensor_type.name)
    print(f"Tensor is stored as {ttype} — showing raw data sample:")
    print(embd_tensor.data[:64])

Tensor is stored as Q6_K — showing raw data sample:
[[ 24 251  76 ...  61 214 128]
 [ 65 155 185 ...  81 181   0]
 [110 135 223 ... 149 206 128]
 ...
 [173 113   7 ... 138 212 128]
 [ 33 204 233 ...  53 212   0]
 [ 48  90 123 ... 164 226   0]]


### 6.3 Looking at Other Weight Tensors

Let's look at a normalization layer weight — these are stored as full floats (not quantized) and are the easiest to inspect directly.

In [25]:
# Find a normalization tensor (usually stored as F32 or F16 — easy to read)
norm_tensor = None
for tensor in reader.tensors:
    if "norm" in tensor.name and "blk.0" in tensor.name:
        norm_tensor = tensor
        break

if norm_tensor is not None:
    print(f"Tensor: {norm_tensor.name}")
    print(f"Shape:  {list(norm_tensor.shape)}")
    print(f"Type:   {norm_tensor.tensor_type.name}")
    print()
    
    norm_data = norm_tensor.data
    print(f"Data dtype: {norm_data.dtype}")
    
    # Convert to float32 for display
    floats = norm_data.astype(np.float32)
    print(f"First 20 weight values:")
    print(np.round(floats[:20], 5))
    print()
    print(f"Statistics: min={floats.min():.5f}, max={floats.max():.5f}, "
          f"mean={floats.mean():.5f}, std={floats.std():.5f}")
    print()
    print("Note: normalization weights are usually close to 1.0 (they start at 1.0 during training)")
else:
    # Fallback: show any tensor with F32 data
    for tensor in reader.tensors:
        if "norm" in tensor.name:
            print(f"Found norm tensor: {tensor.name}")
            print(f"  Shape: {list(tensor.shape)}, Type: {tensor.tensor_type.name}")
            data = tensor.data.astype(np.float32)
            print(f"  First 10 values: {np.round(data[:10], 5)}")
            break

Tensor: blk.0.attn_norm.weight
Shape:  [np.uint64(1536)]
Type:   F32

Data dtype: float32
First 20 weight values:
[0.67578 0.71484 0.46289 0.26172 0.33398 0.59375 0.24609 0.41602 0.58984
 0.27539 0.25781 1.3125  0.4082  0.45703 0.2334  0.51953 0.57422 0.30469
 0.53516 0.2793 ]

Statistics: min=-0.82031, max=5.62500, mean=0.44927, std=0.24379

Note: normalization weights are usually close to 1.0 (they start at 1.0 during training)


---
## 7. How Concepts Map to Numbers: Token Embeddings

### The Embedding Space

Each token maps to a **dense vector** — a list of ~1000–4000 floating-point numbers. These vectors aren't arbitrary: after training, tokens with **similar meanings end up with similar vectors**.

Famous analogy (from Word2Vec, 2013):  
> `vector("king") - vector("man") + vector("woman") ≈ vector("queen")`

For economics, we'd hope something like:  
> `vector("inflation") - vector("price") + vector("quantity") ≈ vector("output")`

We can extract embedding vectors for specific tokens using `llama_cpp`'s embedding mode and visualize them.

### Measuring Token Similarity with Cosine Distance

The standard way to compare embedding vectors is **cosine similarity**: values range from -1 (opposite) to +1 (identical direction in embedding space).

In [26]:
# Load the model in embedding mode to extract token vectors
# Note: we need embedding=True to get per-token vectors
embed_model = Llama(
    model_path=full_model_path,
    embedding=True,
    n_ctx=128,
    verbose=False
)
print("Embedding model ready.")

llama_context: n_ctx_per_seq (128) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
ggml_metal_init: skipping kernel_get_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_set_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_c4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_1row              (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_l4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_bf16                  (not supported)
ggml_metal_init: skipping kernel_mul_mv_id_bf16_f32                (not supported)
ggml_metal_init: skipping kernel_mul_mm_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mm_id_bf16_f16                (not supported)
ggml_metal_init: skipping kernel_flash_attn_ext_bf16_h64  

Embedding model ready.


In [32]:
# Economics words to compare
economics_words = [
    "inflation",
    "deflation",
    "tariff",
    "trade",
    "GDP",
    "recession",
    "growth",
    "supply",
    "demand",
    "price",
]


def collapse_embedding(vec: np.ndarray) -> np.ndarray:
    """Return a single vector even if llama-cpp gives per-token rows."""
    if vec.ndim == 1:
        return vec
    if vec.ndim == 2:
        return vec.mean(axis=0)
    raise ValueError(f"Unexpected embedding shape: {vec.shape}")


# Get embeddings for each word
word_embeddings = {}
for word in economics_words:
    emb = embed_model.embed(word)
    emb_array = np.array(emb, dtype=np.float32)
    word_embeddings[word] = collapse_embedding(emb_array)

# Print the shape of one embedding
sample_word = economics_words[0]
print(f"Embedding for '{sample_word}':")
print(f"  Shape: {word_embeddings[sample_word].shape}")
print(f"  First 10 values: {np.round(word_embeddings[sample_word][:10], 4)}")
print(f"  Min: {word_embeddings[sample_word].min():.4f}")
print(f"  Max: {word_embeddings[sample_word].max():.4f}")
print(f"  Norm (length): {np.linalg.norm(word_embeddings[sample_word]):.4f}")

Embedding for 'inflation':
  Shape: (1536,)
  First 10 values: [ 1.0864  5.6866  1.473  -1.0398 -1.267   2.5275 -3.4899  2.4664  1.8001
 -1.6579]
  Min: -63.3888
  Max: 35.0134
  Norm (length): 146.5680


In [33]:
def cosine_similarity(v1, v2):
    """Cosine similarity between two vectors. Range: -1 (opposite) to +1 (same direction)."""
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 == 0 or n2 == 0:
        return 0.0
    return float(np.dot(v1, v2) / (n1 * n2))

# Build a similarity matrix
words = economics_words
n = len(words)
sim_matrix = np.zeros((n, n))

for i, w1 in enumerate(words):
    for j, w2 in enumerate(words):
        sim_matrix[i, j] = cosine_similarity(word_embeddings[w1], word_embeddings[w2])

# Print the similarity matrix
print("Cosine Similarity Matrix for Economics Words")
print("(1.0 = identical direction, 0.0 = unrelated, -1.0 = opposite)")
print()
print(f"{'':15}", end="")
for w in words:
    print(f"{w:>10}", end="")
print()
print("-" * (15 + 10 * n))
for i, w1 in enumerate(words):
    print(f"{w1:<15}", end="")
    for j in range(n):
        v = sim_matrix[i, j]
        print(f"{v:>10.3f}", end="")
    print()

Cosine Similarity Matrix for Economics Words
(1.0 = identical direction, 0.0 = unrelated, -1.0 = opposite)

                inflation deflation    tariff     trade       GDP recession    growth    supply    demand     price
-------------------------------------------------------------------------------------------------------------------
inflation           1.000     0.778     0.786     0.701     0.523     0.844     0.743     0.767     0.708     0.788
deflation           0.778     1.000     0.665     0.564     0.338     0.709     0.605     0.600     0.556     0.614
tariff              0.786     0.665     1.000     0.752     0.491     0.775     0.722     0.720     0.703     0.792
trade               0.701     0.564     0.752     1.000     0.511     0.673     0.903     0.883     0.877     0.862
GDP                 0.523     0.338     0.491     0.511     1.000     0.533     0.543     0.542     0.536     0.466
recession           0.844     0.709     0.775     0.673     0.533     1.000     

In [34]:
# Find the most similar pairs (excluding self-similarity)
print("Most Similar Word Pairs:")
print("-" * 40)

pairs = []
for i in range(n):
    for j in range(i + 1, n):
        pairs.append((sim_matrix[i, j], words[i], words[j]))

pairs.sort(reverse=True)
for sim, w1, w2 in pairs[:8]:
    print(f"  {w1:12} ↔ {w2:12}  similarity = {sim:.4f}")

print()
print("Least Similar Word Pairs:")
print("-" * 40)
for sim, w1, w2 in pairs[-5:]:
    print(f"  {w1:12} ↔ {w2:12}  similarity = {sim:.4f}")

Most Similar Word Pairs:
----------------------------------------
  growth       ↔ demand        similarity = 0.9232
  growth       ↔ supply        similarity = 0.9136
  trade        ↔ growth        similarity = 0.9026
  supply       ↔ demand        similarity = 0.8958
  trade        ↔ supply        similarity = 0.8826
  trade        ↔ demand        similarity = 0.8774
  supply       ↔ price         similarity = 0.8727
  growth       ↔ price         similarity = 0.8666

Least Similar Word Pairs:
----------------------------------------
  inflation    ↔ GDP           similarity = 0.5232
  trade        ↔ GDP           similarity = 0.5110
  tariff       ↔ GDP           similarity = 0.4912
  GDP          ↔ price         similarity = 0.4656
  deflation    ↔ GDP           similarity = 0.3376


### 7.1 Embedding Arithmetic: Concepts as Directions

One of the most remarkable properties of trained embedding spaces is that **relationships between concepts are encoded as directions in vector space**. Let's test some economic analogies.

In [35]:
# Economics analogy: which word completes the relationship?
# Example: inflation is to price as recession is to ???

def find_closest(query_vec, candidates, word_embeddings, exclude=None):
    """Find the word in candidates whose embedding is closest to query_vec."""
    exclude = exclude or []
    best_sim, best_word = -np.inf, None
    for word in candidates:
        if word in exclude:
            continue
        sim = cosine_similarity(query_vec, word_embeddings[word])
        if sim > best_sim:
            best_sim, best_word = sim, word
    return best_word, best_sim

# Extended word list for analogy search
analogy_words = [
    "inflation", "deflation", "tariff", "trade", "GDP", "recession",
    "growth", "supply", "demand", "price", "interest", "bank",
    "export", "import", "tax", "subsidy", "unemployment", "wages"
]

# Get embeddings for the extended list
for word in analogy_words:
    if word not in word_embeddings:
        emb = embed_model.embed(word)
        word_embeddings[word] = np.array(emb, dtype=np.float32)

# Test: "export" - "trade" + "tax" ≈ ???  (export is to trade as tax is to ?)
analogies = [
    ("export",    "trade",    "tariff",   "import"),     # A:B as C:D
    ("inflation", "price",    "recession", "growth"),
    ("supply",    "producer", "demand",   "consumer"),
]

print("Embedding Arithmetic: A is to B as C is to ???")
print("Formula: vec(A) - vec(B) + vec(C) → find closest word")
print("-" * 60)

for A, B, C, expected in analogies:
    if all(w in word_embeddings for w in [A, B, C]):
        query = word_embeddings[A] - word_embeddings[B] + word_embeddings[C]
        result, sim = find_closest(query, analogy_words, word_embeddings, exclude=[A, B, C])
        print(f"  {A:12} - {B:12} + {C:12} ≈ {result:12}  (similarity: {sim:.3f})")
        print(f"     Expected: {expected}")
        print()
    else:
        missing = [w for w in [A, B, C] if w not in word_embeddings]
        print(f"  Skipped (missing embeddings for: {missing})")

Embedding Arithmetic: A is to B as C is to ???
Formula: vec(A) - vec(B) + vec(C) → find closest word
------------------------------------------------------------


/var/folders/wx/mgl11c114vv7vpz1f_b7szwr0000gn/T/ipykernel_66773/2217016693.py:6: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return float(np.dot(v1, v2) / (n1 * n2))


ValueError: shapes (1,1536) and (1,1536) not aligned: 1536 (dim 1) != 1 (dim 0)

**Note:** Embedding arithmetic works best in larger models with richer training data. Small 1B-parameter models may not perfectly recover every analogy, but the nearest neighbors should still be semantically reasonable. The point is that **relationships between words are encoded as geometric relationships between vectors**.

---
## 8. Putting It All Together: The Full Pipeline

Let's trace one economics sentence all the way through the pipeline to see every numeric transformation.

In [31]:
input_text = "Lower interest rates stimulate investment and economic growth."

print("=" * 70)
print("STEP 1: RAW TEXT")
print("=" * 70)
print(f"  '{input_text}'")
print(f"  Length: {len(input_text)} characters")

print()
print("=" * 70)
print("STEP 2: TOKENIZATION (text → integers)")
print("=" * 70)
token_ids = model.tokenize(input_text.encode("utf-8"))
print(f"  Token IDs: {token_ids}")
print(f"  Count: {len(token_ids)} tokens")

print()
print("=" * 70)
print("STEP 3: TOKEN → TEXT MAPPING")
print("=" * 70)
pieces = [model.detokenize([tid]).decode("utf-8", errors="replace") for tid in token_ids]
print(f"  {'ID':>8}  {'Text piece'}")
for tid, piece in zip(token_ids, pieces):
    print(f"  {tid:>8}  {repr(piece)}")

print()
print("=" * 70)
print("STEP 4: EMBEDDING LOOKUP (each integer → vector of floats)")
print("=" * 70)
print("  (Getting embeddings for each token piece...)")
for tid, piece in zip(token_ids[:5], pieces[:5]):  # show first 5 to keep output manageable
    emb = embed_model.embed(piece.strip() or piece)
    emb_arr = np.array(emb, dtype=np.float32)
    print(f"  Token {tid:>6} {repr(piece):>20}  →  vector shape {emb_arr.shape}")
    print(f"    First 8 values: {np.round(emb_arr[:8], 4)}")
    print(f"    Norm: {np.linalg.norm(emb_arr):.4f}")
if len(token_ids) > 5:
    print(f"  ... ({len(token_ids) - 5} more tokens)")

print()
print("=" * 70)
print("STEP 5: TRANSFORMER LAYERS")
print("=" * 70)
n_layers = sum(1 for t in reader.tensors if "blk." in t.name and "attn_q.weight" in t.name)
print(f"  The model runs {n_layers} transformer blocks in sequence.")
print(f"  Each block applies: attention (Q/K/V) + feed-forward network + layer norm")
print(f"  This is {n_layers * 4}+ matrix multiplications per token per forward pass.")

print()
print("=" * 70)
print("STEP 6: GENERATE NEXT TOKEN")
print("=" * 70)
prompt_tokens = model.tokenize(input_text.encode("utf-8"))
next_tok = next(model.generate(prompt_tokens, top_k=1))
next_text = model.detokenize([next_tok]).decode("utf-8", errors="replace")
print(f"  Most probable next token ID: {next_tok}")
print(f"  Decoded: {repr(next_text)}")
print(f"  Continued sentence: '{input_text}{next_text}...'") 

STEP 1: RAW TEXT
  'Lower interest rates stimulate investment and economic growth.'
  Length: 62 characters

STEP 2: TOKENIZATION (text → integers)
  Token IDs: [9053, 2734, 7813, 49977, 9162, 323, 6955, 6513, 13]
  Count: 9 tokens

STEP 3: TOKEN → TEXT MAPPING
        ID  Text piece
      9053  'Lower'
      2734  ' interest'
      7813  ' rates'
     49977  ' stimulate'
      9162  ' investment'
       323  ' and'
      6955  ' economic'
      6513  ' growth'
        13  '.'

STEP 4: EMBEDDING LOOKUP (each integer → vector of floats)
  (Getting embeddings for each token piece...)
  Token   9053              'Lower'  →  vector shape (1, 1536)
    First 8 values: [[-2.3206  3.9744  1.3143 ...  0.9436  6.3813 -4.1203]]
    Norm: 177.1868
  Token   2734          ' interest'  →  vector shape (1, 1536)
    First 8 values: [[ 3.6182  8.6499  3.114  ...  1.9512  1.2711 -3.0735]]
    Norm: 168.5734
  Token   7813             ' rates'  →  vector shape (1, 1536)
    First 8 values: [[ 5.6946  6

---
## 9. Summary: Numbers All the Way Down

In this notebook, you explored how a language model is entirely a numeric computation:

| Stage | What It Looks Like | The Numbers |
|-------|-------------------|-------------|
| Text input | `"A tariff is a tax on..."` | A Python string |
| After tokenization | Tokens going in | `[362, 287, 31954, ...]` (list of ints) |
| Embedding lookup | Each token → vector | `[-0.0123, 0.0412, ...]` (1000s of floats) |
| Transformer forward pass | Weight matrices × embedding vectors | Billions of float multiplications |
| Output logits | Score for every vocab token | One float per vocab entry (~150k floats) |
| Sampling / decoding | Pick next token ID | One int → decoded back to text |

### Key Takeaways

1. **A vocabulary is just a lookup table**: token IDs are indices into a big table mapping integers to text pieces.

2. **Embeddings are coordinates in meaning-space**: semantically similar words end up with geometrically close vectors after training.

3. **GGUF is a self-describing file**: metadata + weight tensors in one binary file. You can inspect a model's architecture, vocabulary, and weights without ever running it.

4. **Quantization trades precision for space**: Q4_0 stores 32 weights in 18 bytes instead of 128 bytes, with only a small quality penalty.

5. **Generation is sequential**: the model produces one token ID at a time, each time doing a full forward pass through all transformer layers.

### Next Steps

- `Inside_Small_Model.ipynb` — visualize token probabilities and decoding strategies
- `LlamaCpp_SmallLM_Demo.ipynb` — build multi-turn economic policy chatbot
- [The Illustrated Transformer](http://jalammar.github.io/illustrated-transformer/) — visual walkthrough of attention and transformer math
- [GGUF spec](https://github.com/ggerganov/ggml/blob/master/docs/gguf.md) — full binary format documentation

In [ ]:
# Clean up: close model handles to free memory
del model
del embed_model
del reader
print("Models and file handles released.")